# EVASPA Evaporative Fraction processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pprint import pprint

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap

In [ ]:
from sweat.common.io import read_data_from_file
from sweat.evaspa.ef import (
    EFConfig,
    EFModel,
    check_variability,
    initialize,
    run,
)

In [ ]:
def plot_band(data: xr.Dataset, band: str, percentile: float = 0.0):
    data[band].plot.imshow(
        vmax=data[band].quantile(1 - percentile),
        vmin=data[band].quantile(percentile),
    )


def plot_mask(data: xr.Dataset, mask: str, invert=False):
    cmap = [(0.0, 0.0, 0.0, 0.0)]
    if len(np.unique(data[mask].data)) > 1:
        cmap = [(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)]
    if invert:
        cmap = cmap[::-1]
    valid_cmap = ListedColormap(cmap)
    data[mask].plot(cmap=valid_cmap, add_colorbar=False)


def plot_models(
    data: xr.Dataset,
    models: xr.Dataset,
):
    """
    Plot EF models
    """
    # Set figure subplots
    nb = len(models.data_vars)
    row = int(np.ceil(nb / 2))
    col = 2
    fig = plt.figure(figsize=(4 * col, 3 * row), constrained_layout=True)

    i = 0
    j = 0
    # Loop over models
    for model_name in models.data_vars:
        if model_name in ["flags", "valid"]:
            continue
        model = EFModel.create(models[model_name].attrs)
        var_name = model.var
        var = data[model.var].data
        lst = data["lst"].data
        ax = plt.subplot2grid((row, col), (i, j))
        # plot LST points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)
        var_arr = np.linspace(var_min, var_max, num=100)
        tw_arr = model.twet(var_arr)
        td_arr = model.tdry(var_arr)
        ax.plot(
            var_arr,
            td_arr,
            color="tab:red",
            linewidth=3,
            label="dry edge",
            zorder=1,
        )
        ax.plot(
            var_arr,
            tw_arr,
            color="tab:blue",
            linewidth=3,
            label="wet edge",
            zorder=1,
        )
        # Labels
        ax.set_xlabel(var_name)
        ax.set_ylabel("Land Surface Temperature K")
        ax.set_title(f"Model {model_name}", fontsize=12)
        j += 1
        if j == 2:
            j = 0
            i += 1

    # title
    fig.suptitle("Evaporative fraction methods", fontsize=12)

    plt.show()


def plot_data(
    data: xr.Dataset,
    var_name: str,
):
    """
    Plot
    """
    # Figure dimensions
    _ = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    lst = data["lst"].data
    var = data[var_name].data
    # plot all points
    ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    plt.show()


def plot_dataset(
    data: xr.Dataset,
):
    """
    Plot EF models
    """
    # Set figure subplots
    nb = len(data.data_vars)
    row = int(np.ceil(nb / 2))
    col = 2
    fig = plt.figure(figsize=(4 * col, 3 * row), constrained_layout=True)

    i = 0
    j = 0
    # Loop over variables
    for var in data.data_vars:
        ax = plt.subplot2grid((row, col), (i, j))
        if var == "valid":
            valid_cmap = ListedColormap(
                [(1.0, 0.0, 0.0, 1.0), (0.0, 1.0, 0.0, 1.0)]
            )
            data[var].plot(ax=ax, cmap=valid_cmap, add_colorbar=False)
        elif var == "flags":
            da = data[var]
            unique_combinations = np.unique(da.values)
            # Use a qualitative colormap with enough colors
            # (e.g., tab20 or a custom one)
            cmap = plt.get_cmap("tab20", len(unique_combinations))
            # Map each unique mask value to an integer index (for coloring)
            value_to_index = {v: i for i, v in enumerate(unique_combinations)}
            indexed_data = da.copy(
                data=np.vectorize(value_to_index.get)(da.values)
            )
            indexed_data.plot(ax=ax, cmap=cmap, add_colorbar=False)
            legend_patches = [
                mpatches.Patch(color=cmap(i), label=format(v, "08b"))
                for i, v in enumerate(unique_combinations)
            ]
            ax.legend(
                handles=legend_patches,
                title="Flags (bits)",
                bbox_to_anchor=(1, 1),
                loc="upper right",
                fontsize="small",
                fancybox=True,
                framealpha=0.5,
            )
        else:
            data[var].plot(ax=ax)
        j += 1
        if j == 2:
            j = 0
            i += 1
    # title
    fig.suptitle("Dataset", fontsize=12)

    plt.show()

### Get test data 

Read test data

In [ ]:
src_path = os.path.dirname(os.path.dirname(os.getcwd()))
data = read_data_from_file(
    os.path.join(src_path, "tests", "data", "modis_test.tif")
)
data

Plot data

In [ ]:
plot_dataset(data)

### Configuration

In [ ]:
config = {
    "models": "hsm_evaspa",
    "options": {
        "filtering": {
            "albedo": {
                "and": [{"op": ">=", "value": 0.1}, {"op": "<=", "value": 0.3}]
            }
        },
        "selection": False,
        "merging": {
            "merging_method": "mean",
            "uncertainty_method": "interquartile",
        },
    },
}

In [ ]:
# Validate parameters config
ef_config = EFConfig.model_validate(config).model_dump(by_alias=True)

In [ ]:
pprint(ef_config, sort_dicts=False)

### Process evaporative fraction

#### Check variability

In [ ]:
check_variability(data["lst"], threshold=0.02)

#### Compute EF

Initialize

In [ ]:
models, options = initialize(ef_config)

In [ ]:
models

In [ ]:
options

Run

In [ ]:
ef_xr, ef_inst_xr = run(models, data, **options)

In [ ]:
ef_xr

In [ ]:
ef_inst_xr

Visualize results

In [ ]:
plot_dataset(ef_inst_xr)

In [ ]:
plot_dataset(ef_xr)

In [ ]:
plot_models(data, models=ef_xr)